# device initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a

from time import sleep as delay
import datetime
import csv

dm = keithley_dm6500("USB0::0x05E6::0x6500::04651237::INSTR")
ps811 = rigol_dp811a(resource_name="USB0::0x1AB1::0x0E11::DP8H245000515::INSTR")
ps821 = rigol_dp821a(resource_name="USB0::0x1AB1::0x0E11::DP8E261000023::INSTR")

from sc_approval.ch341 import ch341
i2c = ch341(logging=False)

from phy.relay_16ch import relay_box
relay = relay_box(i2c_h=i2c, i2c_a=0x27, logging=False)
relay.init_channels
relay.logging = False

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a
from sc_approval.ch341 import ch341
from phy.relay_16ch import relay_box

from time import sleep as delay
import datetime
import csv

from PyQt5.QtWidgets import *
from PyQt5.QtGui import *
from PyQt5.QtCore import Qt
from PyQt5 import uic
import sys
import yaml

form_class = uic.loadUiType("./sc_approval/main.ui")[0]

class WindowClass(QMainWindow, form_class) :

    def __init__(self, logging=False) :

        super().__init__()
        self.setupUi(self)
        self.setFixedSize(self.size())
        self.setWindowFlags(Qt.Window | Qt.WindowMinimizeButtonHint | Qt.WindowCloseButtonHint)

        self.btn_init.clicked.connect(self.init_equipment)
        self.btn_pre.clicked.connect(self.pre_test)
        self.btn_post.clicked.connect(self.post_test)

        self.log_pre.setFontPointSize(15)
        self.log_post.setFontPointSize(15)

        self.mode = None
        self.logging = logging

        with open("./sc_approval/equipment.yaml") as yaml_id:
            id_list = yaml.safe_load(yaml_id)

        yaml_ps_id = id_list["ps"]
        yaml_dm_id = id_list["dm"]

        self.txt_ps_id.setText(yaml_ps_id)
        self.txt_dmm_id.setText(yaml_dm_id)
    

    def closeEvent(self, event):

        event.accept()
        QApplication.quit()


    def init_equipment(self):

        ps_id = self.txt_ps_id.text()
        dmm_id = self.txt_dmm_id.text()

        if self.logging:
            print(f"equipments id : ps={ps_id}, dmm={dmm_id}")

        self.dmm   = keithley_dm6500(dmm_id)
        self.ps821 = rigol_dp821a(ps_id)
        self.i2c   = ch341(logging=False)
        self.relay = relay_box(i2c_h=self.i2c, i2c_a=0x27, logging=False)

        self.relay.init_channels
        self.relay.logging = False

        self.btn_init.setText(f"Initialization (Done)")
        QApplication.processEvents()
    

    def pre_test(self):

        self.mode = "pre_test"
        count = int(self.txt_no1.text())
        if self.logging:
            print(f"enter to pre-test : mode={self.mode}, count={count}")

        self.run_test(count=count)
        self.mode = None
        if self.logging:
            print(f"exit from pre-test mode : mode={self.mode}")


    def post_test(self):

        self.mode = "post_test"
        count = int(self.txt_no1.text())
        if self.logging:
            print(f"enter to post-test : mode={self.mode}, count={count}")

        self.run_test(count=count)
        self.mode = None
        if self.logging:
            print(f"exit from post-test mode : mode={self.mode}")


    def output_csv(self, output, filename=None):
    
        # output should be the list type

        with open(f"./log/{filename}.csv", "a", newline="") as f:
            write_handler = csv.writer(f)
            write_handler.writerow(output)
    

    def output_log(self, message):

        if self.mode =="pre_test":
            self.log_pre.append(message)
        else:
            self.log_post.append(message)
    

    def run_test(self, count):

        v_vext = 20
        v_vout = 4.5

        now  = datetime.datetime.now()
        date = "[" + now.strftime("%Y-%m-%d") + "]"
        filename = f"{date}_{self.mode}"

        #-----------------------------------------------------
        # need to assign the equipment
        #-----------------------------------------------------
        self.ps = self.ps821.ch1
        self.dm = self.dmm
        #-----------------------------------------------------

        self.dm.current_1
        self.relay.init_channels

        self.ps.disable
        delay(1.5)

        self.relay.ch5.enable # start signal

        #-----------------------------------------------------
        #  IQ_VEXT
        #-----------------------------------------------------
        self.relay.ch3.enable
        self.ps.cfg_all = v_vext, 0.05 # vext
        self.ps.enable
        delay(1)

        ret_00h = self.i2c.i2c_read(110, 0)
        self.output_log(f"[I2C Read] 00h={ret_00h:#04x}")
        QApplication.processEvents()

        iq_vext = round(self.dm.current_10E_3 * 1e+6, 3)
        self.output_log(f"[IQ_VEXT] {iq_vext}uA")
        QApplication.processEvents()

        self.ps.cfg_all = 0.1, 0.02
        self.ps.disable
        delay(0.5)
        self.relay.ch3.disable
        #-----------------------------------------------------

        #-----------------------------------------------------
        #  IQ_VOUT
        #-----------------------------------------------------
        self.relay.ch1.enable
        self.ps.cfg_all = v_vout, 0.05 # vout
        self.ps.enable
        delay(1)

        ret_00h = self.i2c.i2c_read(110, 0)
        self.output_log(f"[I2C Read] 00h={ret_00h:#04x}")
        QApplication.processEvents()

        iq_vout = round(self.dm.current_10E_3 * 1e+6, 3)
        self.output_log(f"[IQ_VOUT] {iq_vout}uA")
        QApplication.processEvents()

        self.ps.cfg_all = 0.1, 0.02
        self.ps.disable
        delay(0.5)
        self.relay.ch1.disable
        #-----------------------------------------------------

        self.output_csv(output=[count, iq_vext, iq_vout], filename=filename)

        self.output_log(f"--- Test done : count {count} ---")
        QApplication.processEvents()

        self.relay.ch5.disable


if __name__ == "__main__" :

    app = QApplication(sys.argv) 
    myWindow = WindowClass() 
    myWindow.show()
    app.exec_()

# test code

In [ ]:
relay.init_channels

In [ ]:
relay.ch1.enable

In [ ]:
relay.ch1.disable

In [ ]:
i2c.smbus_scan()

In [ ]:
for i in range(30):
    print(hex(i2c.i2c_read(110, i)))

In [ ]:
ps821.ch1.vset=4

In [ ]:
ps821.ch1.enable